# Assignment 3 — Bundle Adjustment (Task 1)

## PyTorch Implementation: Bundle Adjustment from Scratch

**Goal**: Recover camera intrinsics (focal length $f$), extrinsics ($R_i$, $T_i$ for 50 views), and 20,000 3D point coordinates from 2D observations via gradient-based optimization.

### 1. Setup & Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

### 2. Configuration & Constants

All cameras share the same intrinsic parameters (PINHOLE model). The image resolution is 1024×1024.

In [ ]:
IMAGE_W, IMAGE_H = 1024, 1024
CX, CY = IMAGE_W / 2, IMAGE_H / 2  # Principal point at image center
NUM_VIEWS = 50
NUM_POINTS = 20000

DATA_DIR = "data"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Image size: {IMAGE_W}×{IMAGE_H}, cx={CX}, cy={CY}")
print(f"Views: {NUM_VIEWS}, Points: {NUM_POINTS}")

### 3. Data Loading

Load 2D observations from `points2d.npz` and per-point RGB colors from `points3d_colors.npy`.

Each observation has shape `(20000, 3)` with columns `(x, y, visibility)`:
- `x, y`: pixel coordinates in the current view
- `visibility`: 1.0 = visible, 0.0 = occluded

In [ ]:
def load_data():
    data = np.load(f"{DATA_DIR}/points2d.npz")
    colors = np.load(f"{DATA_DIR}/points3d_colors.npy")
    all_obs = [data[f"view_{i:03d}"] for i in range(NUM_VIEWS)]
    observations = np.stack(all_obs, axis=0)  # (50, 20000, 3)
    return (
        torch.tensor(observations[:, :, :2], dtype=torch.float32, device=DEVICE),
        torch.tensor(observations[:, :, 2], dtype=torch.float32, device=DEVICE),
        torch.tensor(colors, dtype=torch.float32, device=DEVICE),
    )

obs_2d, visibility, colors = load_data()
total_vis = (visibility > 0.5).sum().item()
print(f"obs_2d shape: {obs_2d.shape}")
print(f"Visible pairs: {total_vis} / {NUM_VIEWS * NUM_POINTS} ({100 * total_vis / (NUM_VIEWS * NUM_POINTS):.1f}%)")

### 4. Data Visualization

Show a few views with their 2D projected points overlaid. Each point's color encodes its index.

In [ ]:
import cv2

# Generate unique color per point index
indices = np.linspace(0, 255, NUM_POINTS, dtype=np.uint8).reshape(1, -1)
colorbar = cv2.applyColorMap(indices, cv2.COLORMAP_HSV)[0]  # (N, 3) BGR
point_colors_bgr = colorbar

sample_views = [0, 12, 25, 37, 49]
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for idx, view_i in enumerate(sample_views):
    # Original image
    img = cv2.imread(f"{DATA_DIR}/images/view_{view_i:03d}.png")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, idx].imshow(img_rgb)
    axes[0, idx].set_title(f"View {view_i:03d}")
    axes[0, idx].axis('off')
    
    # Image with 2D projections overlaid
    obs = obs_2d[view_i].cpu().numpy()
    vis = visibility[view_i].cpu().numpy() > 0.5
    overlay = img.copy()
    for j in range(NUM_POINTS):
        if vis[j]:
            x, y = int(obs[j, 0]), int(obs[j, 1])
            color = tuple(int(c) for c in point_colors_bgr[j])
            cv2.circle(overlay, (x, y), 2, color, -1)
    overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    axes[1, idx].imshow(overlay_rgb)
    axes[1, idx].set_title(f"View {view_i:03d} + Projections")
    axes[1, idx].axis('off')

plt.suptitle("Top: Rendered Images | Bottom: 2D Projections Overlay", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 5. Euler Angles ↔ Rotation Matrix

We parameterize rotation using **Euler angles** (XYZ intrinsic convention).
The rotation matrix is constructed as:

$$R = R_z(\gamma) \cdot R_y(\beta) \cdot R_x(\alpha)$$

This gives 3 parameters per camera (compact, no constraints needed).

In [ ]:
def euler_angles_to_matrix(euler_angles, convention="XYZ"):
    """Convert Euler angles (..., 3) to rotation matrices (..., 3, 3).
    Intrinsic XYZ convention: R = Rz @ Ry @ Rx
    """
    rx, ry, rz = euler_angles[..., 0], euler_angles[..., 1], euler_angles[..., 2]
    ca, sa = torch.cos(rx), torch.sin(rx)
    cb, sb = torch.cos(ry), torch.sin(ry)
    cc, sc = torch.cos(rz), torch.sin(rz)
    z0 = torch.zeros_like(ca)
    o0 = torch.ones_like(ca)

    Rz = torch.stack([
        torch.stack([cc, -sc, z0], dim=-1),
        torch.stack([sc,  cc, z0], dim=-1),
        torch.stack([z0,  z0, o0], dim=-1),
    ], dim=-2)
    Ry = torch.stack([
        torch.stack([cb, z0, sb], dim=-1),
        torch.stack([z0, o0, z0], dim=-1),
        torch.stack([-sb, z0, cb], dim=-1),
    ], dim=-2)
    Rx = torch.stack([
        torch.stack([o0, z0,  z0], dim=-1),
        torch.stack([z0, ca, -sa], dim=-1),
        torch.stack([z0, sa,  ca], dim=-1),
    ], dim=-2)
    return Rz @ Ry @ Rx


def matrix_to_euler_angles(R, convention="XYZ"):
    """Convert rotation matrices back to Euler angles (XYZ)."""
    sy = R[..., 0, 2]  # sin(beta)
    mask = (torch.abs(sy) > 0.99999).float()
    beta = torch.asin(torch.clamp(sy, -1.0, 1.0))
    alpha = torch.atan2(-R[..., 1, 2] * (1 - mask) + R[..., 1, 0] * mask,
                          R[..., 2, 2] * (1 - mask) + R[..., 1, 1] * mask)
    gamma = torch.atan2(-R[..., 0, 1] * (1 - mask) + R[..., 1, 0] * mask,
                         R[..., 0, 0] * (1 - mask) + R[..., 1, 1] * mask)
    return torch.stack([alpha, beta, gamma], dim=-1)

# Quick sanity check
test_euler = torch.tensor([[0.1, 0.2, 0.3]])
R_test = euler_angles_to_matrix(test_euler)
euler_recovered = matrix_to_euler_angles(R_test)
print(f"Original:  {test_euler.numpy()[0]}")
print(f"Recovered: {euler_recovered.numpy()[0]}")
print(f"R is orthonormal: {torch.allclose(R_test @ R_test.transpose(-1, -2), torch.eye(3).unsqueeze(0), atol=1e-5)}")

### 6. Projection Model (Pinhole Camera)

**Camera coordinate transform**: $[X_c, Y_c, Z_c]^T = R \cdot P + T$

**Perspective projection** (with coordinate conventions for $Z_c < 0$):

$$u = -f \cdot \frac{X_c}{Z_c} + c_x, \quad v = f \cdot \frac{Y_c}{Z_c} + c_y$$

- The negative sign in $u$ compensates for $Z_c < 0$ (camera looks toward -Z)
- $v$ doesn't need the extra negative because image $y$ points downward

In [ ]:
def project(points_3d, euler_angles, translations, f):
    """Project 3D points to 2D pixel coordinates.
    
    Args:
        points_3d:  (N, 3) or (B, N, 3) — 3D point coordinates
        euler_angles: (V, 3) — Euler angles per view
        translations: (V, 3) — translation vectors per view
        f: scalar — focal length (shared across all views)
    Returns:
        uv_pred: (V, N, 2) — predicted pixel coordinates
    """
    R = euler_angles_to_matrix(euler_angles)  # (V, 3, 3)
    # Xc: (V, N, 3) = R @ points_3d^T + T
    Xc = torch.einsum("vij,jn->vin", R, points_3d.T) + translations.unsqueeze(-1)
    Xc = Xc.permute(0, 2, 1)  # (V, N, 3)
    X, Y, Z = Xc[..., 0], Xc[..., 1], Xc[..., 2]
    u = -f * X / Z + CX
    v =  f * Y / Z + CY
    return torch.stack([u, v], dim=-1)  # (V, N, 2)

### 7. Parameter Initialization

Good initialization is critical for Bundle Adjustment convergence.

**Strategy**:
- **Focal length $f$**: Initialize at 850 px (FoV ≈ 62°, a reasonable assumption)
- **Camera rotations**: Initialize at identity (Euler angles = 0, all cameras face +Z)
- **Camera translations**: $[0, 0, -d]$ with $d = 2.5$ (cameras in front of the object)
- **3D points**: Back-project visible points from the frontal view using the initial $f$ and $d$

In [ ]:
def init_points_from_frontal(obs_2d, visibility, f_init, d_init):
    """Back-project points from view_000 (near-frontal view)."""
    u0, v0 = obs_2d[0, :, 0], obs_2d[0, :, 1]
    # From projection equations (assuming R=I, T=[0,0,-d_init]):
    # u = -f * X / (-d_init) + cx  =>  X = (u - cx) * d_init / f
    # v =  f * Y / (-d_init) + cy  =>  Y = -(v - cy) * d_init / f
    X = (u0 - CX) * d_init / f_init
    Y = -(v0 - CY) * d_init / f_init
    torch.manual_seed(42)
    Z = 0.05 * torch.randn(NUM_POINTS, device=DEVICE)
    pts = torch.stack([X, Y, Z], dim=-1)
    # For points invisible in the frontal view, use random initialization
    invisible = ~(visibility[0] > 0.5)
    if invisible.any():
        torch.manual_seed(42)
        pts[invisible] = 0.05 * torch.randn(invisible.sum().item(), 3, device=DEVICE)
    return pts

# Demonstrate initialization
f_init = 850.0
d_init = 2.5
pts_init = init_points_from_frontal(obs_2d, visibility, f_init, d_init)
print(f"Initial points shape: {pts_init.shape}")
print(f"Points range: X=[{pts_init[:,0].min():.2f}, {pts_init[:,0].max():.2f}], "
      f"Y=[{pts_init[:,1].min():.2f}, {pts_init[:,1].max():.2f}], "
      f"Z=[{pts_init[:,2].min():.2f}, {pts_init[:,2].max():.2f}]")

### 8. Bundle Adjustment — Multi-Stage Optimization

**Optimization objective** (Mean Squared Reprojection Error):

$$\mathcal{L} = \frac{1}{|\mathcal{V}|} \sum_{(i,j) \in \mathcal{V}} \left\| \pi(P_j; R_i, T_i, f) - \mathbf{obs}_{ij} \right\|^2$$

**Multi-stage strategy**:
1. **Stage 1** (Cameras only): Optimize $f$, $R_i$, $T_i$ with 3D points fixed
2. **Stage 2** (Points only): Optimize $P_j$ with cameras fixed
3. **Stage 3** (Joint): Optimize all parameters together
4. **Stage 4** (Polish): Full-batch fine-tuning with very low learning rates

We use **mini-batch SGD** (5000 points per batch) for efficiency, with **Cosine Annealing Warm Restarts** to escape local minima.

In [ ]:
def run_bundle_adjustment(obs_2d, visibility,
                          f_init=850.0, d_init=2.5,
                          batch_size=5000,
                          epochs_stage1=4000, epochs_stage2=4000, epochs_stage3=6000):
    """Run multi-stage Bundle Adjustment optimization."""
    mask = visibility > 0.5
    all_idx = torch.arange(NUM_POINTS, device=DEVICE)

    # --- Initialize parameters ---
    f = nn.Parameter(torch.tensor(float(f_init), device=DEVICE))
    euler = nn.Parameter(torch.zeros(NUM_VIEWS, 3, device=DEVICE))
    trans = nn.Parameter(torch.tensor([[0.0, 0.0, -float(d_init)]], device=DEVICE).repeat(NUM_VIEWS, 1))
    points_3d = nn.Parameter(init_points_from_frontal(obs_2d, visibility, f_init, d_init))

    print(f"Initial: f={f_init:.1f} px, d={d_init:.1f}")
    loss_history = []

    def compute_batch_loss(pidx):
        """Compute reprojection error on a minibatch of points."""
        pts = points_3d[pidx]
        uv_pred = project(pts, euler, trans, f)
        uv_obs = obs_2d[:, pidx, :]
        m = mask[:, pidx]
        diff = uv_pred - uv_obs
        if m.sum() == 0:
            return torch.tensor(0.0, device=DEVICE, requires_grad=True)
        return (diff[m] ** 2).sum() / m.sum()

    @torch.no_grad()
    def full_loss():
        uv_pred = project(points_3d, euler, trans, f)
        diff = uv_pred - obs_2d
        return (diff[mask] ** 2).sum() / mask.sum()

    # ============================================================
    # Stage 1: Cameras only (f, R, T)
    # ============================================================
    print("Stage 1: Optimizing cameras (f, R, T) ...")
    opt1 = torch.optim.Adam([f, euler, trans], lr=3e-3)
    sched1 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt1, T_0=800, T_mult=2)

    for ep in (pbar := tqdm(range(epochs_stage1))):
        opt1.zero_grad()
        pidx = all_idx[torch.randperm(NUM_POINTS, device=DEVICE)[:batch_size]]
        loss = compute_batch_loss(pidx)
        loss.backward()
        opt1.step()
        sched1.step()
        loss_history.append(loss.item())
        if ep % 800 == 0 or ep == epochs_stage1 - 1:
            full = full_loss()
            pbar.set_description(f"S1 loss: {loss.item():.1f} | full: {full.item():.1f}")

    # ============================================================
    # Stage 2: Points only
    # ============================================================
    print("Stage 2: Optimizing 3D points ...")
    opt2 = torch.optim.Adam([points_3d], lr=5e-3)
    sched2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt2, T_0=1000, T_mult=2)

    for ep in (pbar := tqdm(range(epochs_stage2))):
        opt2.zero_grad()
        pidx = all_idx[torch.randperm(NUM_POINTS, device=DEVICE)[:batch_size]]
        loss = compute_batch_loss(pidx)
        loss.backward()
        opt2.step()
        sched2.step()
        loss_history.append(loss.item())
        if ep % 1000 == 0 or ep == epochs_stage2 - 1:
            full = full_loss()
            pbar.set_description(f"S2 loss: {loss.item():.1f} | full: {full.item():.1f}")

    # ============================================================
    # Stage 3: Joint optimization
    # ============================================================
    print("Stage 3: Joint optimization ...")
    opt3 = torch.optim.Adam([
        {"params": [f, euler, trans], "lr": 5e-4},
        {"params": [points_3d], "lr": 2e-3},
    ])
    sched3 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt3, T_0=1500, T_mult=2)

    for ep in (pbar := tqdm(range(epochs_stage3))):
        opt3.zero_grad()
        pidx = all_idx[torch.randperm(NUM_POINTS, device=DEVICE)[:batch_size]]
        loss = compute_batch_loss(pidx)
        loss.backward()
        opt3.step()
        sched3.step()
        loss_history.append(loss.item())
        if ep % 1500 == 0 or ep == epochs_stage3 - 1:
            full = full_loss()
            pbar.set_description(f"S3 loss: {loss.item():.1f} | full: {full.item():.1f}")

    # ============================================================
    # Stage 4: Full-batch fine-tuning
    # ============================================================
    print("Stage 4: Full-batch fine-tuning ...")
    opt4 = torch.optim.Adam([
        {"params": [f, euler, trans], "lr": 1e-4},
        {"params": [points_3d], "lr": 5e-4},
    ])

    for ep in (pbar := tqdm(range(1000))):
        opt4.zero_grad()
        uv_pred = project(points_3d, euler, trans, f)
        diff = uv_pred - obs_2d
        loss = (diff[mask] ** 2).sum() / mask.sum()
        loss.backward()
        opt4.step()
        loss_history.append(loss.item())
        if ep % 250 == 0 or ep == 999:
            rmse = loss.item() ** 0.5
            pbar.set_description(f"S4 loss: {loss.item():.2f} | RMSE: {rmse:.2f} px")

    return {
        "f": f, "euler_angles": euler, "translations": trans,
        "points_3d": points_3d, "loss_history": loss_history,
    }

# Run the optimization
import time
t0 = time.time()
results = run_bundle_adjustment(obs_2d, visibility)
elapsed = time.time() - t0
print(f"\nTotal optimization time: {elapsed:.1f}s ({elapsed / 60:.1f} min)")

### 9. Loss Curve Visualization

Plot the reprojection error (MSE) over the course of optimization, with stage transitions marked.

In [ ]:
loss_history = results["loss_history"]
total = len(loss_history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Linear scale
ax1.plot(loss_history, linewidth=0.5)
s1_end, s2_end, s3_end = 4000, 8000, 14000
for x, name, color in [(s1_end, "S1→S2", "red"), (s2_end, "S2→S3", "orange"), (s3_end, "S3→S4", "green")]:
    ax1.axvline(x=x, color=color, linestyle="--", alpha=0.5)
    ax1.text(x, ax1.get_ylim()[1] * 0.9 if ax1.get_ylim()[1] > 1000 else 30000, name, fontsize=7, ha="center")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Reprojection Error (MSE)")
ax1.set_title("Bundle Adjustment — Loss Curve")
ax1.grid(True, alpha=0.3)

# Log scale
ax2.plot(loss_history, linewidth=0.5)
ax2.set_yscale("log")
for x, _, color in [(s1_end, "S1→S2", "red"), (s2_end, "S2→S3", "orange"), (s3_end, "S3→S4", "green")]:
    ax2.axvline(x=x, color=color, linestyle="--", alpha=0.5)
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Reprojection Error (MSE, log)")
ax2.set_title("Bundle Adjustment — Loss Curve (log scale)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Initial loss: {loss_history[0]:.1f}")
print(f"Final loss:  {loss_history[-1]:.2f}")

### 10. Results — Optimized Parameters

Inspect the recovered focal length, field of view, and camera extrinsics.

In [ ]:
f_val = results["f"].item()
fov = 2 * np.arctan(IMAGE_H / (2 * f_val)) * 180 / np.pi
euler = results["euler_angles"]
trans = results["translations"]
mask = visibility > 0.5

# Final reprojection error
with torch.no_grad():
    uv_pred = project(results["points_3d"], euler, trans, results["f"])
    diff = uv_pred - obs_2d
    rmse = torch.sqrt((diff[mask] ** 2).sum() / mask.sum())

print("=" * 60)
print("  Optimized Results")
print(f"  Focal length: f = {f_val:.2f} px  →  FoV ≈ {fov:.2f}°")
print(f"  Image size:   {IMAGE_W} × {IMAGE_H},  cx = cy = {CX:.0f}")
print(f"  Views: {NUM_VIEWS},  Points: {NUM_POINTS}")
print(f"  Final reprojection RMSE: {rmse.item():.4f} px")
print("=" * 60)

print("\n  Camera Extrinsics (every 10th view):")
print(f"  {'View':<10} {'Ry (deg)':<12} {'Tx':<10} {'Ty':<10} {'Tz':<10}")
print("  " + "-" * 52)
for i in range(0, NUM_VIEWS, 10):
    e = euler[i].detach().cpu().numpy()
    t = trans[i].detach().cpu().numpy()
    ry_deg = e[1] * 180 / np.pi
    print(f"  {i:03d}       {ry_deg:+7.2f}       {t[0]:+.3f}     {t[1]:+.3f}     {t[2]:+.3f}")

### 11. Error Distribution

Histogram of per-point reprojection errors to understand error distribution.

In [ ]:
with torch.no_grad():
    per_point_error = torch.sqrt(((uv_pred - obs_2d) ** 2).sum(dim=-1))  # (V, N)
    errors = per_point_error[mask].cpu().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(errors, bins=100, edgecolor='none', alpha=0.7)
axes[0].axvline(x=np.median(errors), color='red', linestyle='--', label=f'Median: {np.median(errors):.2f} px')
axes[0].axvline(x=np.mean(errors), color='orange', linestyle='--', label=f'Mean: {np.mean(errors):.2f} px')
axes[0].set_xlabel("Reprojection Error (px)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Reprojection Errors")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(errors, bins=100, edgecolor='none', alpha=0.7, cumulative=True, density=True)
axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50th percentile')
axes[1].axhline(y=0.9, color='orange', linestyle='--', alpha=0.5, label='90th percentile')
axes[1].set_xlabel("Reprojection Error (px)")
axes[1].set_ylabel("Cumulative Fraction")
axes[1].set_title("Cumulative Distribution of Reprojection Errors")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Reprojection error statistics:")
print(f"  Mean:   {np.mean(errors):.2f} px")
print(f"  Median: {np.median(errors):.2f} px")
print(f"  P90:    {np.percentile(errors, 90):.2f} px")
print(f"  P95:    {np.percentile(errors, 95):.2f} px")

### 12. Export Results

Save the reconstructed 3D point cloud as a colored OBJ file (viewable in MeshLab), and save all optimized parameters as a PyTorch checkpoint.

In [ ]:
def save_colored_obj(points_np, colors_np, filepath):
    """Save 3D points as OBJ with per-vertex RGB colors."""
    with open(filepath, "w") as f:
        f.write("# Reconstructed 3D point cloud from Bundle Adjustment\n")
        for i in range(len(points_np)):
            x, y, z = points_np[i]
            r, g, b = colors_np[i]
            f.write(f"v {x:.6f} {y:.6f} {z:.6f} {r:.4f} {g:.4f} {b:.4f}\n")
    print(f"OBJ saved: {filepath} ({len(points_np)} vertices)")

# Save OBJ
pts_np = results["points_3d"].detach().cpu().numpy()
cols_np = colors.cpu().numpy()
save_colored_obj(pts_np, cols_np, f"{OUTPUT_DIR}/reconstructed.obj")

# Save PyTorch checkpoint
torch.save({
    "f": results["f"].detach().cpu(),
    "euler_angles": results["euler_angles"].detach().cpu(),
    "translations": results["translations"].detach().cpu(),
    "points_3d": results["points_3d"].detach().cpu(),
}, f"{OUTPUT_DIR}/ba_parameters.pt")
print(f"Parameters saved to {OUTPUT_DIR}/ba_parameters.pt")

### 13. 3D Point Cloud Visualization

Interactive 3D scatter plot of the reconstructed points using matplotlib.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Subsample for faster rendering
subsample = 2000
idx = np.random.RandomState(42).choice(NUM_POINTS, subsample, replace=False)
pts_sample = pts_np[idx]
cols_sample = cols_np[idx]

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts_sample[:, 0], pts_sample[:, 1], pts_sample[:, 2],
           c=cols_sample, s=5, alpha=0.8, marker='o')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title(f'Reconstructed 3D Point Cloud ({subsample} / {NUM_POINTS} points)')

# Set equal aspect ratio
max_range = np.max([
    np.ptp(pts_sample[:, 0]), np.ptp(pts_sample[:, 1]), np.ptp(pts_sample[:, 2])
]) / 2.0
mid_x, mid_y, mid_z = pts_sample[:, 0].mean(), pts_sample[:, 1].mean(), pts_sample[:, 2].mean()
ax.set_xlim(mid_x - max_range, mid_x + max_range)
ax.set_ylim(mid_y - max_range, mid_y + max_range)
ax.set_zlim(mid_z - max_range, mid_z + max_range)

plt.tight_layout()
plt.show()
print("Tip: Run visualize_gif.py to generate a high-quality rotating GIF animation.")

### 14. Camera Pose Visualization

Show the optimized camera positions in 3D space relative to the object.

In [ ]:
# Camera positions in world coordinates: C = -R^T @ T
with torch.no_grad():
    R_all = euler_angles_to_matrix(euler)  # (V, 3, 3)
    T_all = trans  # (V, 3)
    C_all = -torch.einsum("vij,vj->vi", R_all.transpose(-1, -2), T_all)  # (V, 3)
    C_np = C_all.cpu().numpy()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot sparse point cloud
ax.scatter(pts_sample[:, 0], pts_sample[:, 1], pts_sample[:, 2],
           c='gray', s=1, alpha=0.3)

# Plot camera positions
ax.scatter(C_np[:, 0], C_np[:, 1], C_np[:, 2],
           c='red', s=50, marker='^', label='Cameras')

# Draw camera orientation axes
axis_len = 0.3
for i in range(0, NUM_VIEWS, 5):
    c = C_np[i]
    R = R_all[i].cpu().numpy()
    for j, color in enumerate(['r', 'g', 'b']):
        direction = -R[:, j] * axis_len  # camera looks along -Z_c
        ax.quiver(c[0], c[1], c[2], direction[0], direction[1], direction[2],
                  color=color, alpha=0.6, linewidth=1)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Camera Poses & 3D Point Cloud')
ax.legend()

plt.tight_layout()
plt.show()

### 15. Discussion

**Key observations**:

1. **Convergence behavior**: The four-stage optimization with cyclic LR restarts effectively avoids local minima. The loss decreases from ~37,000 MSE to ~298 MSE (RMSE ≈ 17 px).

2. **Initialization sensitivity**: Starting from a naive initialization (identity rotations, single depth, random Z) makes the problem highly non-convex. The staged approach (cameras → points → joint → polish) helps navigate this.

3. **Comparison to standard SfM**: In practice, SfM pipelines (like COLMAP) use feature matching and incremental reconstruction to get good initial estimates before global BA. Starting directly from 2D observations is a harder problem.

4. **Error sources**: The residual RMSE of ~17 px on 1024×1024 images may come from:
   - Unmodeled distortion effects
   - Non-Gaussian observation noise
   - Local minima in the optimization landscape

**Possible improvements**:
- Use Levenberg-Marquardt solver (standard for BA)
- Better initialization via two-view geometry
- Add robust loss functions (Huber, Cauchy) to handle outliers
- Model radial distortion parameters